# E-Commerce Customer & Sales Intelligence — Python EDA & Analytics

**Phase 2/4 of the portfolio project**

This notebook analyzes the cleaned e-commerce dataset using **Python, Pandas, NumPy, Matplotlib and Seaborn**.

### Business goal
Turn transaction, customer, product, delivery and return data into business insights about:
- sales and profitability
- customer behavior and RFM segmentation
- products and categories
- discounting
- delivery performance
- returns

> **Learning note:** This notebook is completed for the portfolio project, but the explanations are intentionally beginner-friendly. When Python/Pandas reaches these topics in your data-analytics class, you can recreate each section yourself.

## 1. Libraries

- **Pandas** → tables/DataFrames and data manipulation
- **NumPy** → numerical operations
- **Matplotlib / Seaborn** → charts and visual exploration

Think of a Pandas DataFrame as a programmable spreadsheet. Many Pandas operations have a direct SQL equivalent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")

## 2. Load the four datasets

The project has four related tables:

1. `customers` — customer information
2. `products` — product and pricing information
3. `orders` — transaction-level sales
4. `delivery_returns` — delivery and return information

**SQL connection:** `pd.read_csv()` is similar to loading a table into a working dataset; later, `merge()` will play a role similar to SQL `JOIN`.

In [ ]:
DATA_PATH = "data"

customers = pd.read_csv(f"{DATA_PATH}/customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(f"{DATA_PATH}/products.csv")
orders = pd.read_csv(f"{DATA_PATH}/orders.csv", parse_dates=["order_date"])
delivery_returns = pd.read_csv(
    f"{DATA_PATH}/order_delivery_returns.csv",
    parse_dates=["delivery_date"]
)

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Delivery & Returns:", delivery_returns.shape)

## 3. First inspection

We inspect:
- number of rows and columns
- column names
- data types
- sample records
- summary statistics

This is the Python equivalent of the initial SQL checks such as `DESCRIBE`, `SELECT * LIMIT 5`, and basic aggregations.

In [ ]:
print("CUSTOMERS")
display(customers.head())

print("\nPRODUCTS")
display(products.head())

print("\nORDERS")
display(orders.head())

print("\nDELIVERY & RETURNS")
display(delivery_returns.head())

In [ ]:
print("Data types — Orders")
display(orders.dtypes)

print("\nNumeric summary — Orders")
display(orders.describe().T)

## 4. Data-quality validation

The cleaned project files should contain no missing cells and should preserve the relationships between the four tables.

We check:
- missing values
- duplicate primary keys
- foreign-key integrity
- date ranges

This is important because analysis is only trustworthy when the underlying data is valid.

In [ ]:
quality = pd.DataFrame({
    "table": ["customers", "products", "orders", "delivery_returns"],
    "rows": [len(customers), len(products), len(orders), len(delivery_returns)],
    "missing_cells": [
        customers.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        delivery_returns.isna().sum().sum()
    ],
    "duplicate_ids": [
        customers["customer_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        delivery_returns["order_id"].duplicated().sum()
    ]
})
display(quality)

print("Order date range:", orders["order_date"].min().date(), "to", orders["order_date"].max().date())

print("Orders with missing customers:",
      (~orders["customer_id"].isin(customers["customer_id"])).sum())

print("Orders with missing products:",
      (~orders["product_id"].isin(products["product_id"])).sum())

print("Delivery/return records without an order:",
      (~delivery_returns["order_id"].isin(orders["order_id"])).sum())

## 5. Executive KPIs

These KPIs provide a high-level view of the business.

**SQL connection:** `sum()`, `count()`, and calculated columns here correspond closely to SQL aggregate functions and KPI formulas.

In [ ]:
total_orders = orders["order_id"].nunique()
units_sold = orders["quantity"].sum()
total_revenue = orders["sales_amount"].sum()
total_profit = orders["profit"].sum()
aov = total_revenue / total_orders
profit_margin = total_profit / total_revenue * 100

purchasing_customers = orders["customer_id"].nunique()
customer_order_counts = orders.groupby("customer_id")["order_id"].nunique()
repeat_customers = (customer_order_counts > 1).sum()
repeat_purchase_rate = repeat_customers / purchasing_customers * 100

delivered = delivery_returns["delivery_status"].isin(["On Time", "Late"])
delivered_orders = delivered.sum()
late_orders = (delivery_returns["delivery_status"] == "Late").sum()
late_rate = late_orders / delivered_orders * 100
avg_delivery_days = delivery_returns.loc[delivered, "delivery_days"].mean()

returned_orders = (delivery_returns["return_status"] == "Returned").sum()
return_rate = returned_orders / delivered_orders * 100
total_refunds = delivery_returns["refund_amount"].sum()

kpis = pd.DataFrame({
    "KPI": [
        "Total Orders", "Units Sold", "Total Revenue", "Total Profit",
        "AOV", "Profit Margin", "Purchasing Customers",
        "Repeat Customers", "Repeat Purchase Rate",
        "Average Delivery Days", "Late Delivery Rate",
        "Return Rate", "Total Refunds"
    ],
    "Value": [
        total_orders, units_sold, total_revenue, total_profit,
        aov, profit_margin, purchasing_customers,
        repeat_customers, repeat_purchase_rate,
        avg_delivery_days, late_rate, return_rate, total_refunds
    ]
})
display(kpis)

## 6. Sales trends

We examine yearly and monthly revenue/profit.

The purpose is not only to make a chart, but to answer:
- Is the business growing?
- Is profit growing faster or slower than revenue?
- Are there strong/weak months?

**SQL connection:** this is similar to `GROUP BY YEAR(order_date)` and `GROUP BY DATE_FORMAT(order_date, '%Y-%m')`.

In [ ]:
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.to_period("M").astype(str)

yearly = orders.groupby("year").agg(
    orders=("order_id", "nunique"),
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum")
).reset_index()

yearly["revenue_growth_pct"] = yearly["revenue"].pct_change() * 100
yearly["profit_growth_pct"] = yearly["profit"].pct_change() * 100

display(yearly)

In [ ]:
monthly = orders.groupby("month").agg(
    orders=("order_id", "nunique"),
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum")
).reset_index()

display(monthly.sort_values("revenue", ascending=False).head(10))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly["month"], monthly["revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(monthly["month"], monthly["profit"], marker="o")
plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Profit")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 7. Category and product analysis

We join orders with products to analyze category/subcategory/product performance.

**SQL connection:** `merge()` is conceptually similar to a SQL `JOIN`.

In [ ]:
order_products = orders.merge(
    products[["product_id", "product_name", "category", "subcategory", "brand",
              "unit_cost", "list_price"]],
    on="product_id",
    how="left"
)

category = order_products.groupby("category").agg(
    orders=("order_id", "nunique"),
    units=("quantity", "sum"),
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum")
).reset_index()

category["profit_margin_pct"] = category["profit"] / category["revenue"] * 100
category = category.sort_values("revenue", ascending=False)

display(category)

In [ ]:
top_revenue_products = order_products.groupby(
    ["product_id", "product_name"]
).agg(
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum"),
    units=("quantity", "sum")
).reset_index().sort_values("revenue", ascending=False).head(10)

top_profit_products = order_products.groupby(
    ["product_id", "product_name"]
).agg(
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum"),
    units=("quantity", "sum")
).reset_index().sort_values("profit", ascending=False).head(10)

print("Top 10 products by revenue")
display(top_revenue_products)

print("Top 10 products by profit")
display(top_profit_products)

In [ ]:
product_performance = order_products.groupby(
    ["product_id", "product_name", "category"]
).agg(
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum"),
    units=("quantity", "sum")
).reset_index()

high_sales_low_profit = product_performance[
    (product_performance["revenue"] > product_performance["revenue"].quantile(0.75)) &
    (product_performance["profit"] <= 0)
].sort_values("revenue", ascending=False)

display(high_sales_low_profit)

## 8. Discount vs profitability

`discount_pct` is stored as a decimal proportion:
- `0.05` = 5%
- `0.10` = 10%
- `0.20` = 20%

We create discount bands and compare revenue/profit margins.

**Business question:** Does heavier discounting appear to reduce observed profitability?

In [ ]:
def discount_band(x):
    if x == 0:
        return "No Discount"
    elif x <= 0.10:
        return "1-10%"
    elif x <= 0.20:
        return "11-20%"
    elif x <= 0.30:
        return "21-30%"
    return "30%+"

orders["discount_band"] = orders["discount_pct"].apply(discount_band)

discount_analysis = orders.groupby("discount_band").agg(
    orders=("order_id", "nunique"),
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum")
).reset_index()

discount_analysis["profit_margin_pct"] = (
    discount_analysis["profit"] / discount_analysis["revenue"] * 100
)

band_order = ["No Discount", "1-10%", "11-20%", "21-30%", "30%+"]
discount_analysis["sort"] = discount_analysis["discount_band"].map(
    {v: i for i, v in enumerate(band_order)}
)
discount_analysis = discount_analysis.sort_values("sort").drop(columns="sort")

display(discount_analysis)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(discount_analysis["discount_band"], discount_analysis["profit_margin_pct"])
plt.title("Observed Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.show()

## 9. Customer behavior

We calculate:
- orders per customer
- one-time vs repeat customers
- top customers by revenue

This converts transaction-level data into customer-level insights.

In [ ]:
customer_summary = orders.groupby("customer_id").agg(
    orders=("order_id", "nunique"),
    revenue=("sales_amount", "sum"),
    profit=("profit", "sum"),
    units=("quantity", "sum"),
    last_order_date=("order_date", "max")
).reset_index()

customer_summary["customer_type"] = np.where(
    customer_summary["orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

customer_type = customer_summary.groupby("customer_type").agg(
    customers=("customer_id", "nunique"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum")
).reset_index()

top_customers = customer_summary.sort_values(
    "revenue", ascending=False
).head(10)

display(customer_type)
display(top_customers)

## 10. RFM customer segmentation

RFM means:

- **Recency** → how recently the customer purchased
- **Frequency** → how often the customer purchased
- **Monetary** → how much revenue the customer generated

We use the final order date in the dataset (`2025-12-31`) as the analysis reference date.

The scoring uses five groups (1–5). Higher recency score means a more recent purchase; higher frequency/monetary scores mean more purchases/higher value.

**SQL connection:** this is similar to the `NTILE(5)` window-function logic used in the SQL phase.

In [ ]:
analysis_date = orders["order_date"].max()

rfm = orders.groupby("customer_id").agg(
    last_order_date=("order_date", "max"),
    frequency=("order_id", "nunique"),
    monetary=("sales_amount", "sum")
).reset_index()

rfm["recency_days"] = (analysis_date - rfm["last_order_date"]).dt.days

rfm["r_score"] = pd.qcut(
    rfm["recency_days"].rank(method="first"),
    5, labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["m_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["rfm_score"] = (
    rfm["r_score"].astype(str) +
    rfm["f_score"].astype(str) +
    rfm["m_score"].astype(str)
)

display(rfm.head())

In [ ]:
def rfm_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 4:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost Customers"
    else:
        return "Potential Loyalists"

rfm["segment"] = rfm.apply(rfm_segment, axis=1)

rfm_segments = rfm.groupby("segment").agg(
    customers=("customer_id", "nunique"),
    avg_recency_days=("recency_days", "mean"),
    avg_frequency=("frequency", "mean"),
    total_monetary=("monetary", "sum")
).reset_index().sort_values("customers", ascending=False)

display(rfm_segments)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(rfm_segments["segment"], rfm_segments["customers"])
plt.title("RFM Customer Segments")
plt.xlabel("Segment")
plt.ylabel("Customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 11. Delivery and shipping analysis

We compare shipping modes using:
- average delivery days
- late-delivery rate

This helps identify the trade-off between promised speed and execution reliability.

In [ ]:
shipping = delivery_returns.merge(
    orders[["order_id", "shipping_mode"]],
    on="order_id",
    how="left"
)

shipping_analysis = shipping.groupby("shipping_mode").agg(
    orders=("order_id", "nunique"),
    avg_delivery_days=("delivery_days", "mean"),
    late_orders=("delivery_status", lambda x: (x == "Late").sum()),
    delivered_orders=("delivery_status", lambda x: x.isin(["On Time", "Late"]).sum())
).reset_index()

shipping_analysis["late_rate_pct"] = (
    shipping_analysis["late_orders"] /
    shipping_analysis["delivered_orders"] * 100
)

display(shipping_analysis)

## 12. Returns analysis

We examine:
- return rate
- return reasons
- refund impact
- return impact by category

The goal is to distinguish preventable operational/product problems from customer-driven returns.

In [ ]:
return_reasons = delivery_returns[
    delivery_returns["return_status"] == "Returned"
].groupby("return_reason").agg(
    returned_orders=("order_id", "nunique"),
    refunds=("refund_amount", "sum")
).reset_index().sort_values("refunds", ascending=False)

display(return_reasons)

return_category = shipping.merge(
    products[["product_id", "category"]],
    left_on="order_id", right_on="product_id", how="left"
) if False else None

# Correct order-level join for category return analysis
returns_with_category = delivery_returns[
    delivery_returns["return_status"] == "Returned"
].merge(
    orders[["order_id", "product_id"]],
    on="order_id",
    how="left"
).merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

return_by_category = returns_with_category.groupby("category").agg(
    returned_orders=("order_id", "nunique"),
    refunds=("refund_amount", "sum")
).reset_index().sort_values("refunds", ascending=False)

display(return_by_category)

## 13. Key Python findings

The Python analysis reproduces and expands the SQL findings:

1. Overall revenue/profit are strong, but revenue growth between 2024 and 2025 is modest.
2. Home & Kitchen contributes the largest revenue among the major categories.
3. Electronics has a higher observed profit margin than Home & Kitchen.
4. Observed profit margin falls as discount bands increase.
5. Repeat customers form a substantial share of purchasing customers.
6. RFM segmentation reveals Champions, Potential Loyalists, At Risk and Lost Customers that can be targeted differently.
7. Delivery reliability is a major operational issue, especially for faster shipping modes.
8. Quality and damage are important return drivers, creating opportunities for operational improvement.

These findings will feed directly into the **Power BI dashboard** and the final business recommendations.

# Python concepts to learn later in your data-analytics class

You do **not** need to master all of this immediately. Use this project as a reference.

| Project code | Python concept to learn |
|---|---|
| `pd.read_csv()` | Reading files with Pandas |
| `DataFrame.head()` | Inspecting data |
| `isna()` | Missing-value checking |
| `groupby()` | Similar to SQL `GROUP BY` |
| `agg()` | Multiple aggregations |
| `merge()` | Similar to SQL `JOIN` |
| `.dt.year`, `.dt.to_period()` | Date handling |
| `np.where()` | Conditional logic |
| `.apply()` | Applying a function |
| `pd.qcut()` | Quantile-based grouping |
| `matplotlib` | Basic visualization |
| `seaborn` | Statistical visualization |

### Most important connection

You already know SQL.

For example:

**SQL**
```sql
SELECT category, SUM(sales_amount)
FROM orders
GROUP BY category;
```

**Pandas**
```python
orders.groupby("category")["sales_amount"].sum()
```

The analytical idea is the same. Python gives you another tool for doing the analysis and visualization.